# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

In [198]:
# Load environment variables
%load_ext dotenv
%dotenv 
# Add src to path
import os
import sys 
import pandas as pd

# Define the path for the new subdirectory
sys.path.append("../data/fires")


# URL of the Forest Fire data set
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/forest-fires/forestfires.csv"

# Download the data
forestfires = pd.read_csv(url)
forestfires
# Download the data


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00


## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [323]:
# Load the libraries as required.
# Load environment variables
%reload_ext dotenv
%dotenv 
# Add src to path
import os
import sys 
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler, OneHotEncoder, QuantileTransformer,PowerTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score,root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Define the path for the new subdirectory
sys.path.append("../05_src/data/fires")


# URL of the Forest Fire data set
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/forest-fires/forestfires.csv"

# Download the data
fires_dt = pd.read_csv(url)
fires_dt


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00


In [200]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [201]:
import pandas as pd
fires_df = pd.DataFrame(fires_dt)
fires_df

,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00


In [331]:
numeric_features = ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_features = ['month', 'day']
coordinate_features = ['coord_x', 'coord_y']
target_features = ['area_log']
fires_df_upd = fires_df.assign(area_log = np.log(fires_df["area"] + 1))

fires_df_upd



,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain,area,area_log
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00,0.000000
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00,0.000000
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00,0.000000
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00,0.000000
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44,2.006871
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29,4.012592
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16,2.498152
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00,0.000000


# Get X and Y

Create the features data frame and target data.

In [333]:
#identify x and y
x = fires_df_upd[coordinate_features + categorical_features + numeric_features]
y = fires_df_upd[target_features]

#create train and test sets
scoring = ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [398]:

# Simple Preprocessor: Scales numeric variables and recodes categorical variables
#preproc1
preproc1 = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), categorical_features)
    ]
)
preproc1

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh',
                                  'wind', 'rain']),
                                ('categorical',
                                 OneHotEncoder(drop='if_binary',
                                               handle_unknown='ignore'),
                                 ['month', 'day'])])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [399]:
#preproc2
simple_pipeline = Pipeline([
   # ('power_tranformer',PowerTransformer(method='yeo-johnson')),
    ('power_tranformer',PowerTransformer()),
    ('scaler', StandardScaler())
 ])

preproc2 = ColumnTransformer([
    ('numeric', simple_pipeline, numeric_features),
    ('categorical', OneHotEncoder(handle_unknown = 'ignore', drop = 'if_binary'), categorical_features)],
    remainder='drop'
)
preproc2 

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('power_tranformer',
                                                  PowerTransformer()),
                                                 ('scaler', StandardScaler())]),
                                 ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh',
                                  'wind', 'rain']),
                                ('categorical',
                                 OneHotEncoder(drop='if_binary',
                                               handle_unknown='ignore'),
                                 ['month', 'day'])])

In [352]:
transformed_data = preproc2.fit_transform(fires_df)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [400]:
# Pipeline A = preproc1 + baseline
pipe_A = Pipeline(steps =[
('preprocessor', preproc1), 
('regressor', KNeighborsRegressor())
])
pipe_A

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', KNeighborsRegressor())])

In [401]:
# Pipeline B = preproc2 + baseline
pipe_B = Pipeline([('preprocessor', preproc2),
('regressor', KNeighborsRegressor())])
pipe_B

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('power_tranformer',
                                                                   PowerTransformer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', KNeighborsRegressor())])

In [402]:
# Pipeline C = preproc1 + advanced model
pipe_C = Pipeline([('preprocessor', preproc1), 
('regressor', RandomForestRegressor())])
pipe_C

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', RandomForestRegressor())])

In [403]:
# Pipeline D = preproc2 + advanced model
pipe_D = Pipeline([('preprocessor', preproc2), 
('regressor', RandomForestRegressor())])
pipe_D
    

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('power_tranformer',
                                                                   PowerTransformer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', RandomForestRegressor())])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [457]:
pipe_A.get_params()
# Define the parameter grid
knn_param_grid = {'regressor__n_neighbors': range(1, 10, 2)}

#fit the model with different combinations of hyperparaeters in 5 folds
from sklearn.model_selection import train_test_split, GridSearchCV

gridsearch = GridSearchCV(
    estimator=pipe_A,
    param_grid=knn_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    refit= 'neg_root_mean_squared_error'
)
gridsearch.fit(x_train,y_train)
#Find the result of pipe_A model
results_A = pd.DataFrame(gridsearch.cv_results_)
results_A

c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\mi

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.029431,0.006739,0.022494,0.008536,1,{'regressor__n_neighbors': 1},-1.905456,-2.030684,-2.083912,-2.085925,-1.639253,-1.949046,0.168203,5
1,0.041257,0.007855,0.022578,0.009222,3,{'regressor__n_neighbors': 3},-1.581942,-1.746267,-1.724228,-1.750113,-1.524946,-1.665499,0.093668,4
2,0.022925,0.007086,0.016074,0.005143,5,{'regressor__n_neighbors': 5},-1.594631,-1.512568,-1.658170,-1.679794,-1.400043,-1.569041,0.102582,3
3,0.020858,0.004997,0.012085,0.001487,7,{'regressor__n_neighbors': 7},-1.504692,-1.469991,-1.580595,-1.649286,-1.385626,-1.518038,0.090729,2
4,0.024348,0.004549,0.016858,0.004621,9,{'regressor__n_neighbors': 9},-1.502869,-1.434891,-1.550092,-1.631972,-1.340650,-1.492095,0.099250,1


In [458]:
pipe_B.get_params()

# Define the parameter grid
knn_param_grid = {'regressor__n_neighbors': range(1, 10, 2)}

gridsearch = GridSearchCV(
    estimator=pipe_B,
    param_grid=knn_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    refit= 'neg_root_mean_squared_error'
)
gridsearch.fit(x_train,y_train)

#Find the result of pipe_B model
results_B = pd.DataFrame(gridsearch.cv_results_)
results_B

c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\mi

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.116926,0.020428,0.032533,0.015008,1,{'regressor__n_neighbors': 1},-1.971996,-2.054712,-1.980079,-2.103278,-1.627906,-1.947594,0.167071,5
1,0.086188,0.014243,0.016920,0.004143,3,{'regressor__n_neighbors': 3},-1.615228,-1.669357,-1.693951,-1.730682,-1.590818,-1.660007,0.051043,4
2,0.074310,0.016388,0.017906,0.003125,5,{'regressor__n_neighbors': 5},-1.576372,-1.498680,-1.669233,-1.716273,-1.454324,-1.582977,0.098903,3
3,0.074356,0.012686,0.017376,0.005059,7,{'regressor__n_neighbors': 7},-1.517515,-1.426342,-1.625762,-1.646187,-1.406307,-1.524422,0.098707,2
4,0.076457,0.018997,0.018381,0.005703,9,{'regressor__n_neighbors': 9},-1.504461,-1.417473,-1.551846,-1.640865,-1.349025,-1.492734,0.101874,1


In [459]:
pipe_C.get_params()
# Define the parameter grid
rf_param_grid = {'regressor__n_estimators': range(1, 10, 2)}

gridsearch = GridSearchCV(
    estimator=pipe_C,
    param_grid=rf_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    refit= 'neg_root_mean_squared_error'
)
gridsearch.fit(x_train,y_train)

#Find the result of pipe_C model
results_C = pd.DataFrame(gridsearch.cv_results_)
results_C

c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was pa

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.032428,0.005754,0.017535,0.007043,1,{'regressor__n_estimators': 1},-1.708220,-2.122059,-1.978110,-2.208016,-1.886033,-1.980488,0.175994,5
1,0.051772,0.011801,0.019021,0.014280,3,{'regressor__n_estimators': 3},-1.701694,-1.762615,-1.698706,-1.943860,-1.499575,-1.721290,0.142335,4
2,0.048957,0.005526,0.012594,0.003459,5,{'regressor__n_estimators': 5},-1.555981,-1.551021,-1.709572,-1.739520,-1.456708,-1.602560,0.106121,2
3,0.054207,0.002888,0.014353,0.004984,7,{'regressor__n_estimators': 7},-1.624911,-1.457309,-1.707895,-1.815717,-1.416180,-1.604402,0.150233,3
4,0.066964,0.007838,0.012036,0.002168,9,{'regressor__n_estimators': 9},-1.576924,-1.565406,-1.721304,-1.789659,-1.250972,-1.580853,0.185689,1


In [456]:
pipe_D.get_params()
# Define the parameter grid
rf_param_grid = {'regressor__n_estimators': range(1, 10, 2)}
gridsearch = GridSearchCV(
    estimator=pipe_D,
    param_grid=rf_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    refit= 'neg_root_mean_squared_error'
)
gridsearch.fit(x_train,y_train)

#Find the result of pipe_C model
results_D = pd.DataFrame(gridsearch.cv_results_)
results_D

c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was pa

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.073344,0.025705,0.022039,0.016060,1,{'regressor__n_estimators': 1},-1.985819,-1.973446,-2.003852,-2.170058,-1.921098,-2.010855,0.084234,5
1,0.096564,0.039511,0.026429,0.024909,3,{'regressor__n_estimators': 3},-1.722015,-1.699880,-1.727434,-1.865952,-1.461976,-1.695452,0.130636,4
2,0.101239,0.018762,0.017292,0.004274,5,{'regressor__n_estimators': 5},-1.569896,-1.518494,-1.639513,-1.819510,-1.372042,-1.583891,0.146944,3
3,0.111653,0.017804,0.013316,0.002917,7,{'regressor__n_estimators': 7},-1.571417,-1.581244,-1.599729,-1.829535,-1.334262,-1.583237,0.156887,2
4,0.138961,0.048540,0.016277,0.003968,9,{'regressor__n_estimators': 9},-1.496645,-1.515527,-1.657964,-1.757069,-1.404568,-1.566355,0.125205,1


# Evaluate

+ Which model has the best performance?

In [468]:
# Calculate the mean for all columns
mean_fit_time_mean_A = results_A['mean_fit_time'].mean()
mean_fit_time_mean_B = results_B['mean_fit_time'].mean()
mean_fit_time_mean_C = results_C['mean_fit_time'].mean()
mean_fit_time_mean_D = results_D['mean_fit_time'].mean()

print(
    str(mean_fit_time_mean_A) + ' - results_A\n' + 
    str(mean_fit_time_mean_B) + ' - results_B\n' + 
    str(mean_fit_time_mean_C) + ' - results_C\n' + 
    str(mean_fit_time_mean_D) + ' - results_D')


0.027763690948486325 - results_A
0.08564720153808593 - results_B
0.05086554527282715 - results_C
0.10435233116149903 - results_D


In [469]:
# Calculate the mean for all columns
mean_score_time_mean_A = results_A['mean_score_time'].mean()
mean_score_time_mean_B = results_B['mean_score_time'].mean()
mean_score_time_mean_C = results_C['mean_score_time'].mean()
mean_score_time_mean_D = results_D['mean_score_time'].mean()

print(
    str(mean_score_time_mean_A) + ' - results_A\n' + 
    str(mean_score_time_mean_B) + ' - results_B\n' + 
    str(mean_score_time_mean_C) + ' - results_C\n' + 
    str(mean_score_time_mean_D) + ' - results_D')

0.018017721176147458 - results_A
0.020623168945312503 - results_B
0.015107688903808595 - results_C
0.019070491790771485 - results_D


# Export

+ Save the best performing model to a pickle file.

In [505]:

# Fit the pipeline to the training data
pipe_C.fit(x_train, y_train)

c:\Users\User\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', RandomForestRegressor())])

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:

import shap
data_transform = pipe_C.named_steps['preprocessor'].transform(x_test)

In [ ]:
# Use SHAP to explain the model's predictions
explainer = shap.Explainer(pipe_C.named_steps['regressor'], pipe_C.named_steps['preprocessor'].transform(x_test))
shap_values = explainer(data_transform)

#initialize SHAP values for the first observation
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0,:], x_test.iloc[0,:])

# Summarize SHAP values across the entire training set
shap.summary_plot(shap_values, pipe_C.named_steps['preprocessor'].transform(x_test), freature_names = x_test.columns)

shap_values = explainer(data_transform)



*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [x] Created a branch with the correct naming convention.
- [x] Ensured that the repository is public.
- [x] Reviewed the PR description guidelines and adhered to them.
- [x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.